# Extending Datasets

This tutorial shows how to create a custom dataset by extending `BaseDataset`. We'll implement a simple CSV dataset loader as an example.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from alf_core.dataclasses import Candidate, LabelledCandidates, Modality
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig

## 2. Define Custom Dataset

Implement the required `load_dataset()` method to load data from your source.

In [ ]:
class CSVDataset(BaseDataset):
    """Custom dataset that loads data from a CSV file."""

    def __init__(
        self, config: BaseDatasetConfig, csv_path: str, feature_cols: list[str], label_col: str
    ):
        super().__init__(config)
        self.csv_path = csv_path
        self.feature_cols = feature_cols
        self.label_col = label_col

    def load_dataset(self) -> LabelledCandidates:
        """Load dataset from CSV file."""
        # Read CSV
        df = pd.read_csv(self.csv_path)

        # Extract features and labels
        features = df[self.feature_cols].values
        labels = df[self.label_col].values

        # Create Candidate objects
        candidates = [
            Candidate(
                data=row,
                modality=self.config.modality,
                features={"raw": row},  # Store features if needed
            )
            for row in features
        ]

        return LabelledCandidates(candidates=candidates, labels=labels)

    def set_metadata(self) -> None:
        """Optionally set dataset-specific metadata."""
        self.metadata = {
            "source": self.csv_path,
            "num_features": len(self.feature_cols),
            "feature_names": self.feature_cols,
        }

## 3. Configuration Example

In [ ]:
# Create configuration
config = BaseDatasetConfig(
    name="my_csv_dataset",
    modality=Modality.TABULAR,
    seed=42,
    train_ratio=0.6,
    validation_frac=0.2,  # 20% of training data
    test_ratio=0.2,
    split_type="random",
)

## 4. Usage Example

In [ ]:
# Create sample CSV file for demonstration
sample_data = pd.DataFrame({
    "feature1": np.random.randn(100),
    "feature2": np.random.randn(100),
    "target": np.random.rand(100),
})
sample_data.to_csv("/tmp/sample_data.csv", index=False)

# Initialize dataset
dataset = CSVDataset(
    config=config,
    csv_path="/tmp/sample_data.csv",
    feature_cols=["feature1", "feature2"],
    label_col="target",
)

# Setup: loads data and creates splits
dataset.setup()

# Access splits
print(f"Training samples: {len(dataset.train_dataset)}")
print(f"Validation samples: {len(dataset.validation_dataset)}")
print(f"Test samples: {len(dataset.test_dataset)}")
print(f"Candidate pool: {len(dataset.candidate_pool)}")

# Query labels for specific candidates
sample_candidates = dataset.candidate_pool.candidates[:5]
labelled = dataset.query(sample_candidates)
print(f"\nQueried {len(labelled)} candidates with labels")

## Key Points

- **Required method**: `load_dataset()` must return a `LabelledCandidates` object
- **Configuration**: Use `BaseDatasetConfig` to specify splits, seed, and modality
- **Modality**: Set appropriate modality (TABULAR, SEQUENCE, IMAGE, GRAPH, etc.)
- **Setup**: Call `dataset.setup()` to load data and create train/val/test/pool splits
- **Splits**: Access via properties: `train_dataset`, `validation_dataset`, `test_dataset`, `candidate_pool_dataset`
- **Query**: Use `query()` to get labels for candidates (for offline evaluation)
- **Optional**: Override `set_metadata()` to store dataset-specific information
- **Features**: Store precomputed features in `Candidate.features` dict if needed